# GCS 三桶数据清单（独立笔记本）

对应 GCP 项目 **`brats-preprocess`** 下的三个存储桶（名称与控制台一致）：

| 桶名 | 用途（与本仓库预训练约定一致） |
|------|--------------------------------|
| **`brats_preprocessed_npy`** | **Contrast**：四模态 `.t1n/.t1c/.t2w/.t2f.npy`，供后续配对 `pair4` |
| **`flare23`** | **Spatial**：`part1`…`part7` 等目录下单个体积 `.npy` |
| **`rsna_2022_ped_preprocessed_npy`** | **Spatial**：预处理 npy 多在前缀 **`npy2/`** 下（可按实际改 `RSNA_PREFIX`） |

**说明**

- 默认只做 **抽样 / 计数**，避免误触全桶递归；每个桶一格代码里把 **`FULL_LISTING = True`** 再跑可生成完整 `.npy` 列表（耗时与清单费用自负）。
- 输出默认在 **`/content/gcs_list_outputs/`**；需要持久化可复制到 Drive 或下载。
- `np.load` 不能直接读 `gs://`，训练请 **gcsfuse** 或先把路径换成挂载目录。

In [ ]:
# 共用：依赖、认证、项目、gcloud
!pip install -q google-cloud-storage

from google.colab import auth

auth.authenticate_user()

PROJECT_ID = "brats-preprocess"
!gcloud config set project {PROJECT_ID}

import os
import subprocess
from pathlib import Path

OUT_ROOT = Path("/content/gcs_list_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("输出目录:", OUT_ROOT)

## ① `gs://brats_preprocessed_npy`（BraTS / contrast）

生成：`brats_sample.txt` 或 `brats_all_npy.txt`（全量时）。可用于后续脚本配对四模态逗号行。

In [ ]:
# ① brats_preprocessed_npy
BUCKET_BRATS = "gs://brats_preprocessed_npy"
FULL_LISTING = False  # True：递归列出全部 .npy，很慢
SAMPLE_LINES = 100    # FULL_LISTING=False 时用 gsutil | head 抽样

out_dir = OUT_ROOT / "brats_preprocessed_npy"
out_dir.mkdir(parents=True, exist_ok=True)

if FULL_LISTING:
    dst = out_dir / "brats_all_npy.txt"
    !gsutil ls -r '{BUCKET_BRATS}/**/*.npy' > {dst}
    !wc -l {dst}
else:
    dst = out_dir / "brats_sample.txt"
    !gsutil ls -r '{BUCKET_BRATS}/**/*.npy' 2>/dev/null | head -n {SAMPLE_LINES} > {dst}
    !wc -l {dst}
    !head -n 5 {dst}

print("已写入:", dst)

## ② `gs://flare23`（FLARE23 / spatial）

生成：`flare23_sample.txt` 或 `flare23_all_npy.txt`。

In [ ]:
# ② flare23
BUCKET_FLARE = "gs://flare23"
FULL_LISTING = False
SAMPLE_LINES = 100

out_dir = OUT_ROOT / "flare23"
out_dir.mkdir(parents=True, exist_ok=True)

if FULL_LISTING:
    dst = out_dir / "flare23_all_npy.txt"
    !gsutil ls -r '{BUCKET_FLARE}/**/*.npy' > {dst}
    !wc -l {dst}
else:
    dst = out_dir / "flare23_sample.txt"
    !gsutil ls -r '{BUCKET_FLARE}/**/*.npy' 2>/dev/null | head -n {SAMPLE_LINES} > {dst}
    !wc -l {dst}
    !head -n 5 {dst}

print("已写入:", dst)

## ③ `gs://rsna_2022_ped_preprocessed_npy`（RSNA-PED / spatial）

仓库内 Colab 曾使用子前缀 **`npy2/`**。若你桶内结构不同，请修改下一格中的 **`RSNA_PREFIX`**。

In [ ]:
# ③ rsna_2022_ped_preprocessed_npy
BUCKET_RSNA = "gs://rsna_2022_ped_preprocessed_npy"
RSNA_PREFIX = f"{BUCKET_RSNA}/npy2"  # 改为 "" 则扫整桶；或改为实际子目录如 "gs://.../其他"
FULL_LISTING = False
SAMPLE_LINES = 100

out_dir = OUT_ROOT / "rsna_2022_ped_preprocessed_npy"
out_dir.mkdir(parents=True, exist_ok=True)

if FULL_LISTING:
    dst = out_dir / "rsna_ped_all_npy.txt"
    !gsutil ls -r '{RSNA_PREFIX}/**/*.npy' > {dst}
    !wc -l {dst}
else:
    dst = out_dir / "rsna_ped_sample.txt"
    !gsutil ls -r '{RSNA_PREFIX}/**/*.npy' 2>/dev/null | head -n {SAMPLE_LINES} > {dst}
    !wc -l {dst}
    !head -n 5 {dst}

print("已写入:", dst)